# Detroit RMS Crime: Bronze-Ingestion

Dieses Notebook unterstützt **drei Lademodi** über die offizielle ArcGIS-REST-API:
`DELTA` (der Normalfall seit dem 18.9.2026), `FULL` (der ganze Jahrgang) und
`DAILY` (ein Kalenderfenster). Es verwendet die produktive Ingestion aus
`detroit_pipeline`, damit Notebook und CLI nicht zwei unterschiedliche
Implementierungen besitzen.

**Pipeline-Stufe:** ArcGIS REST → unveränderter Bronze-Snapshot als **Parquet**
(ZSTD) + Run-Manifest, dazu die Raw-Schicht mit den gelieferten Antworten.


## Schritt 1 – Projektpfad und Imports

Die Zelle sucht den Projekt-Root sowohl beim Start im Root als auch direkt aus dem Notebook-Ordner.

In [ ]:
from datetime import date, timedelta
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src' / 'crime' / 'detroit_pipeline').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src' / 'crime' / 'detroit_pipeline').exists():
    raise RuntimeError('Projekt-Root mit src/crime/detroit_pipeline nicht gefunden')

sys.path.insert(0, str(PROJECT_ROOT / 'src' / 'crime'))

from detroit_pipeline import ArcGISClient
from detroit_pipeline.detroit_arcgis import (
    build_local_date_window, resolve_daily_window,
    build_delta_where,
    build_updated_where,
    build_year_where,
)
from detroit_pipeline.extract_detroit import (
    delta_start,
    extract_delta,
    extract_updated_window,
    extract_year,
    previous_watermark,
    format_load_summary,
)

## Schritt 2 – Lauf konfigurieren

`LOAD_MODE` wählt zwischen drei Modi:

* **`DELTA`** — der Normalfall seit dem 18.9.2026 und das, was der Gesamtlauf
  benutzt. Der erste Aufruf findet kein Wasserzeichen und lädt den Jahrgang
  vollständig; jeder weitere lädt nur, was sich seit dem höchsten gesehenen
  `updated_at` bewegt hat, minus 14 Tage Rückgriff. Der Jahresfilter auf
  `incident_occurred_at` bleibt dabei stehen.
* **`FULL`** — immer der ganze Jahrgang nach `incident_occurred_at`. Für
  Backfills und für Bestandsabgleiche, mit denen sich Löschungen erkennen
  lassen.
* **`DAILY`** — ein Kalenderfenster nach `updated_at`, ohne Jahresfilter und
  ohne Wasserzeichen.

Ein spätes `updated_at` beweist allein keine fachliche Änderung; die spätere
Silver-Stufe vergleicht die Feldwerte.

> **`RUN_DOWNLOAD` ist absichtlich `False`, und hier mehr als eine
> Höflichkeit.** Anders als
> [`02_chicago_bronze_ingestion.ipynb`](02_chicago_bronze_ingestion.ipynb),
> das in ein Wegwerfverzeichnis unter `$TMPDIR` schreibt, schreibt dieses
> Notebook in den **echten Bestand** unter `data/`. Ein Lauf hier erzeugt einen
> echten, unveränderlichen Bronze-Lauf — und rückt bei `DELTA` das
> Wasserzeichen vor.
>
> Nach SeaweedFS **publiziert** es dagegen nicht: Das tut nur `main()` der CLI.
> Wer den Lauf auch oben haben will, ruft
> `python -m detroit_pipeline.extract_detroit --mode delta --years 2024` auf.


In [ ]:
LOAD_MODE = "DELTA"  # Erlaubt: "DELTA", "FULL" oder "DAILY"
RUN_DOWNLOAD = False

# Fuer DELTA und FULL
YEARS = [2024]

# Nur fuer LOAD_MODE = "DAILY"
DAILY_LOOKBACK_DAYS = 14
DAILY_REFERENCE_DATE = None  # None: abgeschlossener Detroit-Vortag; Datum: letzter eingeschlossener Tag

DATA_ROOT = PROJECT_ROOT / "data"
OUTPUT_DIR = DATA_ROOT / "bronze" / "city=detroit"
RAW_DIR = DATA_ROOT / "raw" / "city=detroit"
BATCH_SIZE = 1000
PAUSE_SECONDS = 0.05

{"load_mode": LOAD_MODE, "run_download": RUN_DOWNLOAD, "output_dir": OUTPUT_DIR}


## Schritt 3 – ArcGIS-Metadaten und erwartete Zeilenzahl prüfen

Vor dem Download werden Schema-Fingerprint, Object-ID-Feld, API-Limit und serverseitiger Count geprüft. Im Daily-Modus ist das Intervall halb-offen und verwendet `updated_at`, damit auch nachträgliche Korrekturen erfasst werden.

In [ ]:
client = ArcGISClient()

layer_info = client.get_layer_info()

load_mode = LOAD_MODE.strip().upper()
if load_mode not in {"DELTA", "FULL", "DAILY"}:
    raise ValueError('LOAD_MODE muss "DELTA", "FULL" oder "DAILY" sein')
if DAILY_LOOKBACK_DAYS <= 0:
    raise ValueError("DAILY_LOOKBACK_DAYS muss größer als 0 sein")

source_checks = []
daily_window = resolve_daily_window(DAILY_REFERENCE_DATE, DAILY_LOOKBACK_DAYS)
daily_window_end = daily_window.end_local.date()
daily_window_start = daily_window.start_local.date()
daily_run_date = daily_window_end - timedelta(days=1)  # fachliches Referenzdatum

if load_mode in {"DELTA", "FULL"}:
    for year in YEARS:
        fenster = build_local_date_window(date(year, 1, 1), date(year + 1, 1, 1))
        if load_mode == "DELTA":
            # Das Wasserzeichen kommt aus den eigenen Bronze-Manifesten, nicht
            # aus der Quelle. Ohne eines ist der Lauf ein Vollschnappschuss.
            watermark = previous_watermark(DATA_ROOT, year)
            ab = delta_start(watermark)
            where = build_delta_where(year, ab)
        else:
            watermark, ab = None, None
            where = build_year_where(year)
        source_checks.append({
            "mode": load_mode,
            "year": year,
            "watermark": watermark,
            "delta_ab_utc": ab.isoformat() if ab else None,
            "erster_lauf": ab is None,
            **fenster.manifest_fields(),
            "source_count": client.get_count(where),
        })
else:
    where = build_updated_where(daily_window_start, daily_window_end)
    source_checks.append({
        "mode": "DAILY",
        "date_field": "updated_at",
        **daily_window.manifest_fields(),
        "window_start": daily_window_start.isoformat(),
        "window_end_exclusive": daily_window_end.isoformat(),
        "source_count": client.get_count(where),
    })

layer_check = {
    "object_id_field": layer_info.object_id_field,
    "max_record_count": layer_info.max_record_count,
    "field_count": len(layer_info.fields),
    "schema_fingerprint": layer_info.schema_fingerprint,
}

{"layer": layer_check, "windows": source_checks}


## Schritt 4 – Bronze-Lauf ausführen

Der Extractor fixiert zuerst alle Object-IDs und lädt anschliessend genau diese
IDs batchweise. `DELTA` und `FULL` werden nach `full/year=…` partitioniert,
`DAILY` nach `incremental/run_date=…`. Erst nach bestandener Count-Prüfung
werden Raw und Bronze gemeinsam veröffentlicht — Raw zuerst.


In [ ]:
run_results = []

if RUN_DOWNLOAD and load_mode == "DELTA":
    for year in YEARS:
        year_output_dir = OUTPUT_DIR / "full" / f"year={year}"
        print(f"\nStarte Delta-Ingestion für {year} ...")

        bronze_path, manifest_path = extract_delta(
            year=year,
            output_dir=year_output_dir,
            data_root=DATA_ROOT,
            raw_dir=RAW_DIR / "full" / f"year={year}",
            batch_size=BATCH_SIZE,
            pause_seconds=PAUSE_SECONDS,
        )

        run_results.append({
            "mode": "DELTA",
            "period": str(year),
            "bronze_path": bronze_path,
            "manifest_path": manifest_path,
        })

        print(f"Bronze:  {bronze_path}")
        print(f"Manifest: {manifest_path}")
elif RUN_DOWNLOAD and load_mode == "FULL":
    for year in YEARS:
        year_output_dir = OUTPUT_DIR / "full" / f"year={year}"
        print(f"\nStarte Bronze-Ingestion für {year} ...")

        bronze_path, manifest_path = extract_year(
            year=year,
            output_dir=year_output_dir,
            batch_size=BATCH_SIZE,
            pause_seconds=PAUSE_SECONDS,
        )

        run_results.append({
            "mode": "FULL",
            "period": str(year),
            "bronze_path": bronze_path,
            "manifest_path": manifest_path,
        })

        print(f"Bronze:  {bronze_path}")
        print(f"Manifest: {manifest_path}")
elif RUN_DOWNLOAD and load_mode == "DAILY":
    daily_output_dir = (
        OUTPUT_DIR / "incremental" / f"run_date={daily_run_date.isoformat()}"
    )
    print(
        f"\nStarte Daily-Ingestion: {daily_window_start} "
        f"bis {daily_window_end} (exklusiv) ..."
    )
    bronze_path, manifest_path = extract_updated_window(
        start_date=daily_window_start,
        end_date_exclusive=daily_window_end,
        output_dir=daily_output_dir,
        batch_size=BATCH_SIZE,
        pause_seconds=PAUSE_SECONDS,
    )
    run_results.append({
        "mode": "DAILY",
        "period": f"{daily_window_start}/{daily_window_end}",
        "bronze_path": bronze_path,
        "manifest_path": manifest_path,
    })
    print(f"Bronze:  {bronze_path}")
    print(f"Manifest: {manifest_path}")
else:
    print("Download deaktiviert. Setze RUN_DOWNLOAD = True.")

## Schritt 5 – Geladene Datensätze je Deliktkategorie prüfen

Die Kategoriesumme muss exakt der Gesamtzahl der geladenen Bronze-Zeilen entsprechen. Fehlende Kategorien werden explizit als `__MISSING__` ausgewiesen.

In [ ]:
import json

for result in run_results:
    manifest = json.loads(
        result["manifest_path"].read_text(encoding="utf-8")
    )

    print(f"\n{'=' * 60}")
    print(f"Modus: {result['mode']}")
    print(f"Zeitraum: {result['period']}")
    print(
        f"Selektion: {manifest['selection_basis']} "
        f"über {manifest['date_field']}"
    )
    print(f"Zeilengranularität: {manifest['record_granularity']}")
    print(f"Source-Löschungen erfasst: {manifest['captures_source_deletions']}")
    print(f"Bronze-Datei: {result['bronze_path']}")
    print(f"{'=' * 60}")

    print(
        format_load_summary(
            manifest["offense_category_counts"],
            manifest["downloaded_count"],
        )
    )

## Ergebnis

Nach erfolgreichem Lauf liegen je Lauf zwei Artefakte vor:

- `full/year=<JAHR>/run_<UUID>/part-0001.parquet` für `DELTA` und `FULL`
- `incremental/run_date=<DATUM>/run_<UUID>/part-0001.parquet` für `DAILY`
- je Lauf ein `manifest.json`; Daten und Manifest werden gemeinsam
  veröffentlicht, die Raw-Antworten liegen unter `data/raw/city=detroit/`.

**Ein Delta-Lauf ist kein Bestand.** Er enthält nur die Zeilen, deren
`updated_at` sich seit dem Wasserzeichen bewegt hat, plus die vier Zeilen ohne
Stempel. Der Jahrgang entsteht erst aus dem Vollschnappschuss darunter und
allen Deltas darüber — Silver legt sie über `incident_entry_id` zusammen und
behält je Schlüssel die Fassung mit dem jüngsten `updated_at`.

Die 14 Tage Rückgriff erzeugen absichtlich Überlappung: Bronze bewahrt jede
geladene Fassung mit eigener Run-ID. Gelöschte Datensätze erkennt weder das
Delta noch der Daily Load — dafür braucht es zwei Vollschnappschüsse
(`--mode full`).


## Schritt 6 – Optionaler DuckDB-Bronze-Import

Dieser Import prüft Manifest, Schema und Prüfsumme, bevor er etwas schreibt.
Dabei gibt es drei Ausgänge:

| Fall | Ergebnis |
|---|---|
| neuer Lauf | `IMPORTED` — die Zeilen werden geschrieben, danach gegen `downloaded_count` gezählt |
| bekannter Lauf, gleiche Prüfsumme | `SKIPPED` — nichts wird doppelt geschrieben |
| bekannter Lauf, **andere** Prüfsumme | **Abbruch** — dieselbe Run-ID mit anderen Bytes ist ein Widerspruch, kein Update |

Unterschiedliche überlappende Läufe bleiben nebeneinander erhalten. Das ist
**keine** Silver-Konsolidierung — dafür gibt es `silver_pipeline`.

Die Datenbank ist eine eigene Spielwiese neben der Strecke; die
Auswertungsumgebung des Projekts entsteht in `serving/`. Keine zweite
Anwendung darf dieselbe Datei gleichzeitig schreibend öffnen.


In [ ]:
from detroit_pipeline.load_duckdb import load_run
LOAD_DUCKDB = False
DATABASE = DATA_ROOT / 'warehouse' / 'detroit_bronze.duckdb'
if LOAD_DUCKDB:
    for result in run_results:
        print(load_run(result['manifest_path'], DATABASE))
